<a href="https://colab.research.google.com/github/annnu20/annnu20/blob/main/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
faqs= " " "The production-grade deep reinforcement learning pipeline encountered significant covariance shift during the deployment phase, requiring an immediate transition to an automated MLOps retraining loop. The engineering team initiated a distributed data ingestion workflow that processed approximately 14,850 raw audio samples per second, scaling across an 8-node GPU cluster utilizing mixed-precision arithmetic (FP16/FP32). To address severe class imbalance within the minority target variables, researchers implemented an advanced Synthetic Minority Over-sampling Technique (SMOTE) combined with a customized focal loss function, which successfully boosted the model's macro-averaged recall from 0.64 to 0.89. Ultimately, the system stabilized after 148 epochs of training, reducing cross-entropy loss to 0.124 and establishing a robust inference latency benchmark of 12.8 milliseconds under a concurrent load of 10,000 API requests.  " " "

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
tokenizer = Tokenizer()

In [4]:
tokenizer.fit_on_texts([faqs])


In [5]:
len(tokenizer.word_index)

113

In [8]:
input_sequences = []
for sentence in faqs.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])

In [9]:
input_sequences

[[1, 10],
 [1, 10, 11],
 [1, 10, 11, 12],
 [1, 10, 11, 12, 13],
 [1, 10, 11, 12, 13, 14],
 [1, 10, 11, 12, 13, 14, 15],
 [1, 10, 11, 12, 13, 14, 15, 16],
 [1, 10, 11, 12, 13, 14, 15, 16, 17],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22, 23],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22, 23, 2],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22, 23, 2, 24],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22, 23, 2, 24, 25],
 [1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 1, 21, 22, 23, 2, 24, 25, 3],
 [1,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  1,
  21,
  22,
  23,
  2,
  24,
  25,
  3,
  2],
 [1,
  10,
  11,


In [10]:
max_len = max([len(x) for x in input_sequences])

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_len, padding='pre')

In [12]:
padded_input_sequences

array([[  0,   0,   0, ...,   0,   1,  10],
       [  0,   0,   0, ...,   1,  10,  11],
       [  0,   0,   0, ...,  10,  11,  12],
       ...,
       [  0,   0,   1, ...,   6, 110, 111],
       [  0,   1,  10, ..., 110, 111, 112],
       [  1,  10,  11, ..., 111, 112, 113]], dtype=int32)

In [13]:
X = padded_input_sequences[:,:-1]

In [16]:
y = padded_input_sequences[:,-1]

In [15]:
X.shape

(133, 133)

In [17]:
y.shape

(133,)

In [18]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y,num_classes=283)

In [19]:
y.shape

(133, 283)

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [21]:
model = Sequential()
model.add(Embedding(283, 100, input_length=56))
model.add(LSTM(150, return_sequences=True))
model.add(LSTM(150))
model.add(Dense(283, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(X,y,epochs=10)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 327ms/step - accuracy: 0.0000e+00 - loss: 5.6451
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 325ms/step - accuracy: 0.0526 - loss: 5.6178
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 361ms/step - accuracy: 0.0376 - loss: 5.4141
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 543ms/step - accuracy: 0.0376 - loss: 5.0811
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 322ms/step - accuracy: 0.0376 - loss: 4.9191
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 321ms/step - accuracy: 0.0301 - loss: 4.8110
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 321ms/step - accuracy: 0.0376 - loss: 4.7585
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 327ms/step - accuracy: 0.0301 - loss: 4.7259
Epoch 9/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 636ms/step - accuracy: 0.0301 - loss: 4.6888
Epoch 10/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 326ms/step - accuracy: 0.0376 - loss: 4.6688


In [25]:
import time
import numpy as np
text = "what is the fee"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 426ms/step
what is the fee the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
what is the fee the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
what is the fee the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
what is the fee the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
what is the fee the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
what is the fee the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
what is the fee the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
what is the fee the the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
what is the fee the the the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
what is the fee the the the the the the the the the the


In [26]:
tokenizer.word_index

{'the': 1,
 'an': 2,
 'to': 3,
 'a': 4,
 '0': 5,
 'of': 6,
 '8': 7,
 'minority': 8,
 'loss': 9,
 'production': 10,
 'grade': 11,
 'deep': 12,
 'reinforcement': 13,
 'learning': 14,
 'pipeline': 15,
 'encountered': 16,
 'significant': 17,
 'covariance': 18,
 'shift': 19,
 'during': 20,
 'deployment': 21,
 'phase': 22,
 'requiring': 23,
 'immediate': 24,
 'transition': 25,
 'automated': 26,
 'mlops': 27,
 'retraining': 28,
 'loop': 29,
 'engineering': 30,
 'team': 31,
 'initiated': 32,
 'distributed': 33,
 'data': 34,
 'ingestion': 35,
 'workflow': 36,
 'that': 37,
 'processed': 38,
 'approximately': 39,
 '14': 40,
 '850': 41,
 'raw': 42,
 'audio': 43,
 'samples': 44,
 'per': 45,
 'second': 46,
 'scaling': 47,
 'across': 48,
 'node': 49,
 'gpu': 50,
 'cluster': 51,
 'utilizing': 52,
 'mixed': 53,
 'precision': 54,
 'arithmetic': 55,
 'fp16': 56,
 'fp32': 57,
 'address': 58,
 'severe': 59,
 'class': 60,
 'imbalance': 61,
 'within': 62,
 'target': 63,
 'variables': 64,
 'researchers': 65,
